# Solution 01 — first producer

Reference answer for [`exercise_01_produce_single.ipynb`](../03_exercise/exercise_01_produce_single.ipynb). Read the exercise first; the comments here focus on *why* each line looks the way it does.

In [ ]:
from confluent_kafka import Producer
import json, time

BROKER = 'redpanda:29092'

# A single dict configures the producer. Many more knobs exist; we
# only override what we care about.
producer = Producer({
    'bootstrap.servers': BROKER,
    'client.id':         'python-producer',
})
print(f'Producer connected to {BROKER}')

## Step 2 — build the event

In [ ]:
topic_name    = 'strom'
message_key   = 'haus_a'
message_value = json.dumps({
    'sensor':    'strom',
    'haus':      'haus_a',
    'wert':      42.5,        # any reasonable kWh reading
    'einheit':   'kWh',       # electricity unit
    'timestamp': time.time(), # UNIX seconds
})
print(message_value)

## Step 3 — send + flush

Note `.encode('utf-8')` on key and value — librdkafka takes raw bytes. If you forget you'll get `TypeError: expected bytes`.

In [ ]:
def delivery_report(err, msg):
    if err:
        print(f'Delivery failed: {err}')
    else:
        print(f'Delivered to {msg.topic()} [{msg.partition()}] '
              f'offset {msg.offset()}')

producer.produce(
    topic_name,
    key=message_key.encode('utf-8'),
    value=message_value.encode('utf-8'),
    callback=delivery_report,
)
# flush() blocks until ALL queued messages have been ack'ed by the
# broker. Without it, the program could end before delivery happened.
producer.flush()

## Task A — second key on the same topic

Notice the partition number — it depends on the hash of the key. `haus_a` and `haus_b` typically land in different partitions, but with only 1 or 2 partitions on the topic you sometimes get a hash collision and they share a partition.

In [ ]:
value_b = json.dumps({
    'sensor': 'strom', 'haus': 'haus_b', 'wert': 38.1,
    'einheit': 'kWh', 'timestamp': time.time(),
})
producer.produce('strom', key=b'haus_b', value=value_b.encode(),
                 callback=delivery_report)
producer.flush()

## Task B — different topic

In [ ]:
value_w = json.dumps({
    'sensor': 'wasser', 'haus': 'haus_a', 'wert': 120.5,
    'einheit': 'Liter', 'timestamp': time.time(),
})
producer.produce('wasser', key=b'haus_a', value=value_w.encode(),
                 callback=delivery_report)
producer.flush()

## Task C — broken broker

The error message is typically `KafkaError{code=_MSG_TIMED_OUT, …}` after our 4-second timeout. In production you would log this, count the failure rate (alerting if it spikes), and possibly buffer the message to a local dead-letter file or retry topic for human inspection.

In [ ]:
errors = []
def error_cb(err, msg):
    errors.append(str(err))

p_broken = Producer({
    'bootstrap.servers':  'localhost:9999',
    'message.timeout.ms': 4000,
    'socket.timeout.ms':  2000,
})
p_broken.produce('strom', key=b'haus_a', value=b'{"test": true}',
                 callback=error_cb)
p_broken.flush(timeout=5)
print(errors[0] if errors else 'unexpectedly delivered')